[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Joins &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds the notebook's two tables in a database in memory, and defines `show`, which
prints a query's rows as a table. Run it first. The tasks do not depend on one another, and the last
cell closes the connection.


In [1]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript("""
    CREATE TABLE Departments (
        DeptID   INTEGER PRIMARY KEY,
        DeptName TEXT NOT NULL
    );
    CREATE TABLE Employees (
        EmpID     INTEGER PRIMARY KEY,
        Name      TEXT NOT NULL,
        DeptID    INTEGER REFERENCES Departments (DeptID),
        ManagerID INTEGER REFERENCES Employees (EmpID),
        Salary    INTEGER NOT NULL
    );
    INSERT INTO Departments (DeptID, DeptName) VALUES (1, 'HR'), (2, 'IT'), (3, 'Marketing');
    INSERT INTO Employees (EmpID, Name, DeptID, ManagerID, Salary) VALUES
        (1, 'Alice', 1, NULL, 72000),
        (2, 'Bob', 2, 1, 64000),
        (3, 'Charlie', NULL, 1, 51000);
""")


def show(sql):
    """Run a query and print its rows as a table, with NULL for a missing value."""
    cursor = conn.execute(sql)
    names = [column[0] for column in cursor.description]
    rows = [["NULL" if value is None else str(value) for value in row] for row in cursor]
    widths = [max(len(text) for text in column) for column in zip(names, *rows)]
    for line in [names, ["-" * width for width in widths], *rows]:
        print("  ".join(text.ljust(width) for text, width in zip(line, widths)).rstrip())
    print(f"({len(rows)} row{'' if len(rows) == 1 else 's'})")


print("tables:", [name for (name,) in conn.execute("SELECT name FROM sqlite_schema WHERE type = 'table' ORDER BY name")])


tables: ['Departments', 'Employees']


**1.** Every department with its employees.


In [2]:
show("""
    SELECT Departments.DeptName, Employees.Name
    FROM Departments
    LEFT JOIN Employees ON Employees.DeptID = Departments.DeptID
    ORDER BY Departments.DeptID, Employees.Name
""")


DeptName   Name
---------  -----
HR         Alice
IT         Bob
Marketing  NULL
(3 rows)


Naming `Departments` first makes it the left table, so every department stays, and Marketing, with no
employee to match, has `NULL` for a name. It is the notebook's right join, written as a left join.


**2.** The departments with at least one employee, two ways.


In [3]:
show("""
    SELECT DeptName
    FROM Departments
    WHERE EXISTS (SELECT 1 FROM Employees WHERE Employees.DeptID = Departments.DeptID)
    ORDER BY DeptName
""")
print()
show("""
    SELECT DISTINCT Departments.DeptName
    FROM Departments
    INNER JOIN Employees ON Employees.DeptID = Departments.DeptID
    ORDER BY Departments.DeptName
""")


DeptName
--------
HR
IT
(2 rows)

DeptName
--------
HR
IT
(2 rows)


Both return HR and IT. The inner join makes a row for every employee in a department, so a department
with two employees would appear twice without `DISTINCT`. `EXISTS` only asks whether a match exists,
so it never repeats a department, and needs no `DISTINCT`.


**3.** Every employee beside their manager.


In [4]:
show("""
    SELECT e.Name AS employee,
           m.Name AS manager,
           m.Salary AS manager_salary,
           m.Salary - e.Salary AS earns_less_by
    FROM Employees AS e
    LEFT JOIN Employees AS m ON m.EmpID = e.ManagerID
    ORDER BY e.Name
""")


employee  manager  manager_salary  earns_less_by
--------  -------  --------------  -------------
Alice     NULL     NULL            NULL
Bob       Alice    72000           8000
Charlie   Alice    72000           21000
(3 rows)


The left join keeps Alice, who has no manager, and arithmetic with `NULL` gives `NULL`, so Alice's
difference is `NULL` as well, not 72,000.


**4.** The departments an employee does not work in.


In [5]:
show("""
    SELECT Employees.Name, Departments.DeptName
    FROM Employees
    CROSS JOIN Departments
    WHERE Employees.DeptID IS NOT Departments.DeptID
    ORDER BY Employees.Name, Departments.DeptID
""")


Name     DeptName
-------  ---------
Alice    IT
Alice    Marketing
Bob      HR
Bob      Marketing
Charlie  HR
Charlie  IT
Charlie  Marketing
(7 rows)


Seven of the nine pairs: every pair except Alice in HR and Bob in IT. `IS NOT` treats `NULL` as a
value, so Charlie's `NULL` differs from every `DeptID`. With `!=`, every comparison with Charlie's
`NULL` would have been `NULL`, and the answer four pairs, with Charlie's three missing.


**5.** Dana, in a full join, then rolled back.


In [6]:
conn.execute("INSERT INTO Employees (EmpID, Name, DeptID, ManagerID, Salary) VALUES (4, 'Dana', 3, 2, 58000)")
show("""
    SELECT Employees.Name, Departments.DeptName
    FROM Employees
    LEFT JOIN Departments ON Employees.DeptID = Departments.DeptID
    UNION ALL
    SELECT Employees.Name, Departments.DeptName
    FROM Departments
    LEFT JOIN Employees ON Employees.DeptID = Departments.DeptID
    WHERE Employees.EmpID IS NULL
    ORDER BY Name NULLS LAST
""")
conn.rollback()

print()
print("employees after the rollback:", conn.execute("SELECT COUNT(*) FROM Employees").fetchone()[0])


Name     DeptName
-------  ---------
Alice    HR
Bob      IT
Charlie  NULL
Dana     Marketing
(4 rows)

employees after the rollback: 3


Dana matched Marketing, so the second query, which looks for the departments no employee matched,
returned nothing, and the full join has no row without a name. The insert began a transaction, as an
insert does, and `rollback` undid it, which leaves the three employees the other tasks expect.


**6.** Head count and highest salary for every department.


In [7]:
show("""
    SELECT Departments.DeptName,
           COUNT(Employees.EmpID) AS employees,
           MAX(Employees.Salary) AS highest_salary
    FROM Departments
    LEFT JOIN Employees ON Employees.DeptID = Departments.DeptID
    GROUP BY Departments.DeptID
    ORDER BY Departments.DeptID
""")


DeptName   employees  highest_salary
---------  ---------  --------------
HR         1          72000
IT         1          64000
Marketing  0          NULL
(3 rows)


`COUNT(Employees.EmpID)` gives Marketing 0, where `COUNT(*)` would have counted its row of `NULL`s as
one employee. `MAX` of no salaries is `NULL`, since Marketing has no highest salary to give.

Last, close the connection:


In [8]:
conn.close()


---

&#8592; **Back to:** [Joins](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/05-joins.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
